In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from itertools import product
import sys
sys.path.append('../')
from model_classes import GAT
from helper_functions import create_data, train_one_epoch_GAT, evaluate_GAT, predict_GAT, compute_metrics

In [2]:
adj_matrix = torch.tensor(np.load('../data_folder/Adjacency_matrix.npy'), dtype=torch.float32)
open_prices = pd.read_csv('../data_folder/open_prices_interp.csv', index_col=0)

x = open_prices.values

X_train, y_train, X_val, y_val, X_test, y_test, sc, train_loader, val_loader, test_loader = create_data(
    x, batch_size=32, flatten_data=False, flatten_time_features=True
)

torch.Size([984, 460, 40])
torch.Size([984, 460])


In [3]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
search_space = {
    'c_hidden': [16, 32],
    'num_heads': [2, 4],
    'alpha': [0.2],
    'lr': [1e-3],
}

num_epochs = 5
patience = 2

input_dim = X_train.shape[-1]  # 40
num_relations = adj_matrix.shape[-1]  # 3

keys = list(search_space.keys())
param_grid = [dict(zip(keys, values)) for values in product(*(search_space[k] for k in keys))]
print(f'Running {len(param_grid)} trials')

Running 4 trials


In [4]:
results = []
best = {'best_val_mse': float('inf')} #tracks the single best model seen so far 

for trial_idx, params in enumerate(param_grid, start=1): #loops through all hyperparameter combinations
    
    # num_heads must divide c_hidden cleanly
    if params['c_hidden'] % params['num_heads'] != 0:
        continue

    torch.manual_seed(seed)
    np.random.seed(seed)
    #create a fresh GAT with this trial's hyperparameters
    model = GAT(    
        c_in=input_dim,
        c_hidden=params['c_hidden'],
        c_out=1,
        num_relations=num_relations,
        num_heads=params['num_heads'],
        alpha=params['alpha']
    )

    optimizer = optim.Adam(model.parameters(), lr=params['lr'])
    criterion = nn.MSELoss()

    best_val_mse = float('inf')
    best_state_dict = None
    patience_counter = 0

    print(f"Trial {trial_idx:02d}/{len(param_grid)} | {params}")

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch_GAT(model, train_loader, adj_matrix, optimizer, criterion)
        val_mse = evaluate_GAT(model, val_loader, adj_matrix, criterion)

        print(f"  Epoch {epoch:02d} | train loss {train_loss:.6f} | val MSE {val_mse:.6f}")

        #if val MSE improves we save the model weights and reset the counter. if it gets worse 2 times in a row we stop early
        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_state_dict = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("  Early stopping")
                break

    results.append({**params, 'best_val_mse': best_val_mse})

    if best_val_mse < best['best_val_mse']:
        best = {
            'best_val_mse': best_val_mse,
            'params': params.copy(),
            'state_dict': best_state_dict,
        }

results_df = pd.DataFrame(results).sort_values('best_val_mse').reset_index(drop=True)
display(results_df)

Trial 01/4 | {'c_hidden': 16, 'num_heads': 2, 'alpha': 0.2, 'lr': 0.001}
  Epoch 01 | train loss 6.404091 | val MSE 16.461942
  Epoch 02 | train loss 3.313128 | val MSE 14.951608
  Epoch 03 | train loss 2.386500 | val MSE 11.500353
  Epoch 04 | train loss 1.947279 | val MSE 10.060456
  Epoch 05 | train loss 1.701948 | val MSE 8.812874
Trial 02/4 | {'c_hidden': 16, 'num_heads': 4, 'alpha': 0.2, 'lr': 0.001}
  Epoch 01 | train loss 5.881326 | val MSE 19.009148
  Epoch 02 | train loss 3.309113 | val MSE 15.053438
  Epoch 03 | train loss 2.450031 | val MSE 12.261292
  Epoch 04 | train loss 2.020970 | val MSE 10.802613
  Epoch 05 | train loss 1.777534 | val MSE 9.946078
Trial 03/4 | {'c_hidden': 32, 'num_heads': 2, 'alpha': 0.2, 'lr': 0.001}
  Epoch 01 | train loss 21.103415 | val MSE 40.443674
  Epoch 02 | train loss 8.235912 | val MSE 36.746800
  Epoch 03 | train loss 5.075853 | val MSE 36.709064
  Epoch 04 | train loss 3.780284 | val MSE 29.554872
  Epoch 05 | train loss 2.986910 | val M

,c_hidden,num_heads,alpha,lr,best_val_mse
0,16,2,0.2,0.001,8.812874
1,16,4,0.2,0.001,9.946078
2,32,4,0.2,0.001,14.297375
3,32,2,0.2,0.001,26.183035


In [5]:
# reload best model
best_model = GAT(
    c_in=input_dim,
    c_hidden=best['params']['c_hidden'],
    c_out=1,
    num_relations=num_relations,
    num_heads=best['params']['num_heads'],
    alpha=best['params']['alpha']
)
best_model.load_state_dict(best['state_dict'])

# get predictions
pred_test = predict_GAT(best_model, test_loader, adj_matrix)
pred_test_np = pred_test.numpy()

# compute metrics
metrics = compute_metrics(y_test, pred_test_np)
print("Best params:", best['params'])
print("Test metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

Best params: {'c_hidden': 16, 'num_heads': 2, 'alpha': 0.2, 'lr': 0.001}
Test metrics:
  accuracy: 0.5007
  f1: 0.4997
  mcc: 0.0020
  return_ratio: 0.1018
  sharpe: 0.0089
  MSE: 2.9394


In [6]:
import os

save_dir = '../Best_model'
os.makedirs(save_dir, exist_ok=True)

torch.save({
    'params': best['params'],
    'state_dict': best['state_dict'],
    'metrics': metrics,
}, os.path.join(save_dir, 'GAT_best_model.pth'))

print("Model saved to ../Best_model/GAT_best_model.pth")

Model saved to ../Best_model/GAT_best_model.pth
